In [1]:
import pickle

import pandas as pd
import os
import uuid
import mlflow

from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

In [2]:
from sklearn.pipeline import make_pipeline

In [3]:
year = 2023
month = 2
taxi_type = 'green'

input_file = f'https://d37ci6vzurychx.cloudfront.net/trip-data/{taxi_type}_tripdata_{year:04d}-{month:02d}.parquet'
output_file = f'output/{taxi_type}/{year:04d}-{month:02d}_predicted.parquet'
MLFLOW_TRACKING_URI = "http://ec2-3-80-40-111.compute-1.amazonaws.com:5000/"
RUN_ID = os.getenv("RUN_ID","e0b68d8dd70d4e6fbcaf656cf45a31d5") #Environmental variable e0b68d8dd70d4e6fbcaf656cf45a31d5

In [4]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("nyc-taxi-exp")


# logged_model = f'runs:/{RUN_ID}/model'



<Experiment: artifact_location='s3://mlflow-artifact-mmichal/1', creation_time=1750111858773, experiment_id='1', last_update_time=1750111858773, lifecycle_stage='active', name='nyc-taxi-exp', tags={}>

In [5]:
def generate_uuids(n):
    return [str(uuid.uuid4()) for i in range(n)]

def read_dataframe(filename: str):
    df = pd.read_parquet(filename)
    df['ride_id'] = generate_uuids(df.shape[0])

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60
    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    return df


def prepare_dictionaries(df: pd.DataFrame):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    dicts = df[categorical + numerical].to_dict(orient='records')
    return dicts

In [6]:
def load_model(run_id):
    logged_model = f"s3://mlflow-artifact-mmichal/2/{run_id}/artifacts/model"
    model = mlflow.pyfunc.load_model(logged_model)
    return model

def apply_model(input_file, run_id, output_file):
    df = read_dataframe(input_file)
    dicts = prepare_dictionaries(df)
    
    # Load model as a PyFuncModel.
    model = load_model(run_id) # logged_model
    y_pred = model.predict(dicts)
    df_results = pd.DataFrame()#y_pred
    df_results['ride_id'] = df['ride_id']
    df_results['lpep_dropoff_datetime'] = df.lpep_dropoff_datetime
    df_results['PULocationID'] = df.PULocationID
    df_results['DOLocationID'] = df.DOLocationID
    df_results['actual_duration'] = df.duration
    df_results['predicted_duration'] = y_pred
    df_results['model_version'] = run_id
    df_results['diff'] = df_results['actual_duration'] - y_pred

    df_results.to_parquet(output_file, index=False)
    # df_results.head()

In [7]:
apply_model(input_file, RUN_ID, output_file)